# Comparing Models & Building a Hybrid (Voting) Model — Wine Dataset

1. Tunes **KNN**, **Logistic Regression**, and **SVM** separately with `GridSearchCV`
2. Compares their cross-validated and test accuracy side by side
3. Combines the three tuned models into a **soft-voting hybrid model**
4. Compares the hybrid against each individual model
5. Saves the best individual model and the hybrid model with `joblib`


In [4]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import VotingClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

## 1. Load the data and split

In [5]:
df = pd.read_csv('wine.csv')

X = df.drop('Wine', axis=1)
y = df['Wine']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((142, 13), (36, 13))

## 2. Tune each model individually

In [6]:
# ---- KNN ----
knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])
knn_grid = {'knn__n_neighbors': [3, 5, 7, 9, 11]}

knn_search = GridSearchCV(knn_pipeline, knn_grid, cv=5, scoring='accuracy', n_jobs=-1)
knn_search.fit(X_train, y_train)

print("Best KNN params:", knn_search.best_params_)
print("Best KNN CV accuracy: {:.4f}".format(knn_search.best_score_))

Best KNN params: {'knn__n_neighbors': 11}
Best KNN CV accuracy: 0.9722


In [7]:
# ---- Logistic Regression ----
logreg_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('logreg', LogisticRegression(max_iter=5000, random_state=42))
])
logreg_grid = {'logreg__C': [0.01, 0.1, 1, 10, 100]}

logreg_search = GridSearchCV(logreg_pipeline, logreg_grid, cv=5, scoring='accuracy', n_jobs=-1)
logreg_search.fit(X_train, y_train)

print("Best Logistic Regression params:", logreg_search.best_params_)
print("Best Logistic Regression CV accuracy: {:.4f}".format(logreg_search.best_score_))

Best Logistic Regression params: {'logreg__C': 0.1}
Best Logistic Regression CV accuracy: 0.9931


In [8]:
# ---- SVM ----
svm_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(probability=True, random_state=42))
])
svm_grid = {
    'svm__C': [0.1, 1, 10, 100],
    'svm__kernel': ['linear', 'rbf'],
    'svm__gamma': ['scale', 'auto']
}

svm_search = GridSearchCV(svm_pipeline, svm_grid, cv=5, scoring='accuracy', n_jobs=-1)
svm_search.fit(X_train, y_train)

print("Best SVM params:", svm_search.best_params_)
print("Best SVM CV accuracy: {:.4f}".format(svm_search.best_score_))

Best SVM params: {'svm__C': 0.1, 'svm__gamma': 'scale', 'svm__kernel': 'linear'}
Best SVM CV accuracy: 0.9862


C:\Users\Uswa Khalil\anaconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


## 3. Compare the three tuned models on the test set

In [9]:
models = {
    'KNN': knn_search.best_estimator_,
    'Logistic Regression': logreg_search.best_estimator_,
    'SVM': svm_search.best_estimator_,
}

comparison_rows = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    comparison_rows.append({
        'Model': name,
        'Best CV Accuracy': {
            'KNN': knn_search.best_score_,
            'Logistic Regression': logreg_search.best_score_,
            'SVM': svm_search.best_score_,
        }[name],
        'Test Accuracy': accuracy_score(y_test, y_pred)
    })

comparison_df = pd.DataFrame(comparison_rows).sort_values('Test Accuracy', ascending=False)
comparison_df

,Model,Best CV Accuracy,Test Accuracy
0,KNN,0.972167,1.000000
1,Logistic Regression,0.993103,1.000000
2,SVM,0.986207,0.972222


## 4. soft-voting


In [10]:
voting_model = VotingClassifier(
    estimators=[
        ('knn', knn_search.best_estimator_),
        ('logreg', logreg_search.best_estimator_),
        ('svm', svm_search.best_estimator_),
    ],
    voting='soft'
)

voting_model.fit(X_train, y_train)

voting_cv_scores = cross_val_score(voting_model, X_train, y_train, cv=5, scoring='accuracy')
print("Hybrid model CV accuracy: {:.4f} (+/- {:.4f})".format(
    voting_cv_scores.mean(), voting_cv_scores.std()
))

C:\Users\Uswa Khalil\anaconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Uswa Khalil\anaconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Uswa Khalil\anaconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Hybrid model CV accuracy: 0.9931 (+/- 0.0138)


C:\Users\Uswa Khalil\anaconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Uswa Khalil\anaconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\Uswa Khalil\anaconda3\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [19]:
sample_idx = 0
sample = X_test.iloc[[sample_idx]]
actual_class = y_test.iloc[sample_idx]

knn_probs = knn_search.best_estimator_.predict_proba(sample)[0]
logreg_probs = logreg_search.best_estimator_.predict_proba(sample)[0]
svm_probs = svm_search.best_estimator_.predict_proba(sample)[0]

class_labels = knn_search.best_estimator_.classes_
prob_table = pd.DataFrame(
    [knn_probs, logreg_probs, svm_probs],
    columns = [f'P(Class {c})' for c in class_labels],
    index = ['KNN', 'logistic Regression', 'SVM']
)

prob_table.loc['Soft Voting (avg)'] = prob_table.mean()

print(f"Actual class: {actual_class}")
print(f"Hybrid model prediction: {voting_model.predict(sample)[0]}\n")
prob_table.round(4)

Actual class: 1
Hybrid model prediction: 1



,P(Class 1),P(Class 2),P(Class 3)
KNN,1.0000,0.0000,0.0000
logistic Regression,0.9875,0.0107,0.0018
SVM,0.9969,0.0018,0.0013
Soft Voting (avg),0.9948,0.0042,0.0010


## 5. Compare the hybrid model against the individual models

In [11]:
y_pred_voting = voting_model.predict(X_test)
voting_test_acc = accuracy_score(y_test, y_pred_voting)

final_comparison = pd.concat([
    comparison_df,
    pd.DataFrame([{
        'Model': 'Hybrid (Soft Voting)',
        'Best CV Accuracy': voting_cv_scores.mean(),
        'Test Accuracy': voting_test_acc
    }])
], ignore_index=True).sort_values('Test Accuracy', ascending=False)

final_comparison

,Model,Best CV Accuracy,Test Accuracy
0,KNN,0.972167,1.000000
1,Logistic Regression,0.993103,1.000000
3,Hybrid (Soft Voting),0.993103,1.000000
2,SVM,0.986207,0.972222


In [12]:
print("Classification report — Hybrid (Soft Voting) model:\n")
print(classification_report(y_test, y_pred_voting))

Classification report — Hybrid (Soft Voting) model:

              precision    recall  f1-score   support

           1       1.00      1.00      1.00        12
           2       1.00      1.00      1.00        14
           3       1.00      1.00      1.00        10

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36



In [30]:
#
knn_preds = knn_search.best_estimator_.predict(X_test)
logreg_preds = logreg_search.best_estimator_.predict(X_test)
svm_preds = svm_search.best_estimator_.predict(X_test)
voting_preds = voting_model.predict(X_test)

disagreement_df = pd.DataFrame({
    'KNN': knn_preds,
    'Logistic Regression': logreg_preds,
    'SVM': svm_preds,
    'Hybrid': voting_preds,
    'Actual': y_test.values
}, index = X_test.index)

disagreement_df['Models Agree?'] = disagreement_df[['KNN', 'Logistic Regression', 'SVM']].nunique(axis = 1) == 1
disagreements = disagreement_df[~disagreement_df['Models Agree?']]

print(f"{len(disagreements)} out of { len (X_test)} test samples had disagreement among the base models.\n")
disagreements

1 out of 36 test samples had disagreement among the base models.



,KNN,Logistic Regression,SVM,Hybrid,Actual,Models Agree?
130,3,3,2,3,3,False


## 6. 

In [13]:
best_individual_name = comparison_df.iloc[0]['Model']
best_individual_model = models[best_individual_name]

joblib.dump(best_individual_model, 'best_individual_model.joblib')
joblib.dump(voting_model, 'hybrid_voting_model.joblib')

print(f"Saved best individual model ({best_individual_name}) to best_individual_model.joblib")
print("Saved hybrid soft-voting model to hybrid_voting_model.joblib")

Saved best individual model (KNN) to best_individual_model.joblib
Saved hybrid soft-voting model to hybrid_voting_model.joblib
